In [1]:
import sqlite3

conn = sqlite3.connect("law.db")
cursor = conn.cursor()

In [22]:
conn.close()

In [2]:
def extract_from_db(id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [5]:
rows = extract_from_db(166, include_parent=True)
law = ""
for r in rows:
    law += r['content'] + "\n"
print(law)

QUY TẮC GIAO THÔNG ĐƯỜNG BỘ
Vượt xe và nhường đường cho xe xin vượt
Không được vượt xe trong trường hợp sau đây:
Trên cầu hẹp có một làn đường;



In [7]:
import py_vncorenlp
if "rdrsegmenter" not in globals():
    import py_vncorenlp
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos", "ner"],
        save_dir=r"E:\Github\LawAssistant\test\VnCoreNLP-master"
    )

In [ ]:
rdrsegmenter.close()

In [9]:
import re
import json

text = "Không được vượt xe trong trường hợp sau đây: Trên cầu hẹp có một làn đường;"

def clean_text(text: str) -> str:
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    return text.strip().lower()

text = clean_text(text)
print(text)
output = rdrsegmenter.annotate_text(text)

concepts = []
for sent_id, sent in output.items():
    current = []
    for word in sent:
        token, pos = word["wordForm"], word["posTag"]
        if pos.startswith("N"):   # Nếu là danh từ
            current.append(token)
        else:
            if current:
                concepts.append(" ".join(current))
                current = []
    if current:
        concepts.append(" ".join(current))

print("Concepts extracted:")
for c in concepts:
    print("-", c)

không được vượt xe trong trường hợp sau đây: trên cầu hẹp có một làn đường;
Concepts extracted:
- xe
- trường_hợp sau
- cầu
- làn_đường
